In [8]:
# Import python modules
import os, glob
import geopandas as gpd
import pandas as pd
import pyproj
import numpy as np
import fiona
import time
from geojson import Feature, Point, FeatureCollection
from shapely.geometry import shape, mapping
import scipy.spatial
import json

import rasterio
import rasterio.fill
from rasterstats import zonal_stats

from functools import reduce
from shapely.geometry import Point, Polygon, MultiPoint
from shapely.ops import nearest_points

#import datapane as dp
#!datapane login --token="yourpersonaltoken"
#!datapane login --token="9bde41bfbc4ad14119e32086f9f06d2e5db1d5b8"

import folium
from folium.features import GeoJsonTooltip
import branca.colormap as cm
import os
from IPython.display import display, Markdown, HTML, FileLink, FileLinks

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import datetime

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [11]:
def calculate_demand(row):
    energy_intensity = unique_values_mine_gdf.loc[(unique_values_mine_gdf['iso3']==row['iso3']) & 
                                                  (unique_values_mine_gdf['reference_mineral']==row['reference_mineral']) & 
                                                  (unique_values_mine_gdf['processing_stage']==row['processing_stage']), 'processing_energy_intensity'].values
    return float(row['production_tonnes_for_water'] * 1000 * energy_intensity)

In [14]:
# Read unique values dictionary from an Excel file
unique_values_mine_gdf = pd.read_excel("min_processing_values_dict_New.xlsx")
unique_values_mine_gdf.rename(columns={'isocc': 'iso3', 'mineral': 'reference_mineral', 'stage_cut': 'processing_stage'}, inplace=True)

summary_df = pd.DataFrame()

# Define path and name of the file
mine_path = r"water/water_usage_by_location"
mine_name = ["combined_water_country_constrained", "combined_water_country_unconstrained",
             "combined_water_region_constrained", "combined_water_region_unconstrained"]

summary_df = pd.DataFrame()
for name in mine_name:
    summary_df = pd.DataFrame()
    layers = glob.glob(f'water/water_usage_by_location/{name}*')

    for layer in layers:
        # Create a new geo-dataframe
        mine_gdf = pd.read_csv(layer)
        scenario = layer.split('\\')[-1].replace(name + '_', '') .replace('.csv', '')

        mine_gdf['mining_demand'] = mine_gdf.apply(calculate_demand, axis=1)
        mine_gdf['mining_demand'] = mine_gdf['mining_demand']/1000000 ## GWh/year

        summary = mine_gdf.groupby('iso3').agg({'mining_demand': ['sum']})
        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.rename(columns={'mining_demand_sum': scenario})
        
        # Append the results as a new column in the summary_df
        if summary_df.empty:
            summary_df = summary
        else:
            summary_df = pd.concat([summary_df, summary], axis=1)

    os.makedirs('water/energy_results', exist_ok=True)
    summary_df.to_csv(f"water/energy_results/{name}.csv")
    del summary_df

    # ################   RUN #########################
    
    #     #mine_gdf['mining_demand'] = mine_gdf.apply(calculate_demand_country, axis=1)
    #     mine_gdf['mining_demand'] = mine_gdf.apply(calculate_demand_region, axis=1)
    
    # ################################################
    
    #     mine_gdf['mining_demand'] = mine_gdf['mining_demand']/1000000 ## GWh/year
    
    #     summary = mine_gdf.groupby('iso3').agg({'mining_demand': ['sum']})
    #     summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    #     summary = summary.rename(columns={'mining_demand_sum': layer})
        
    #     # Append the results as a new column in the summary_df
    #     if summary_df.empty:
    #         summary_df = summary
    #     else:
    #         summary_df = summary_df.join(summary, how='outer')

In [9]:
## Function to calculate demand_country
#def calculate_demand_country(row):
#    total_demand = 0
#    for (isocc, mineral, stage_cut), energy_intensity in result_dict.items():
#        #conversion_col = f"{mineral}_conversion_location_in_country"
#        #highest_stage_col = f"{mineral}_mine_highest_stage"
#        #mine_type_col = "mode"
#        production_col = f"{mineral}_final_stage_production_tons_{stage_cut}_in_country"
#        if production_col in row:
#            total_demand += (row[production_col] * 1000) * energy_intensity     
#    return total_demand
#
#def calculate_demand_region(row):
#    total_demand = 0
#    for (mineral, stage_cut), energy_intensity in result_dict.items():
#        #conversion_col = f"{mineral}_conversion_location_in_region"
#        #highest_stage_col = f"{mineral}_mine_highest_stage"
#        #mine_type_col = "mode"
#        production_col = f"{mineral}_final_stage_production_tons_{stage_cut}_in_region"
#        
#        if production_col in row:
#            total_demand += (row[production_col] * 1000) * energy_intensity
#
#    return total_demand

In [10]:
# Function to calculate demand_country for the new format
# def calculate_demand_country(row):
#     total_demand = 0
#     country_code = row['iso3']  # or row['country'] if you use full name

#     for (c, mineral, stage_cut), energy_intensity in result_dict.items():
#         if c != country_code:
#             continue
#         production_col = f"{mineral}_final_stage_production_tons_{stage_cut}_in_country"
#         if production_col in row:
#             total_demand += (row[production_col] * 1000) * energy_intensity
#     return total_demand

# # Function to calculate demand_country for the new format
# def calculate_demand_region(row):
#     total_demand = 0
#     country_code = row['iso3']  # or row['country'] if you use full name

#     for (c, mineral, stage_cut), energy_intensity in result_dict.items():
#         if c != country_code:
#             continue
#         production_col = f"{mineral}_final_stage_production_tons_{stage_cut}_in_region"
#         if production_col in row:
#             total_demand += (row[production_col] * 1000) * energy_intensity
#     return total_demand

In [1]:
# def calculate_demand_country(row):
#     total_demand = 0
#     demand_by_combination = {}
    
#     for (mineral, stage_cut), energy_intensity in result_dict.items():
#         production_col = f"{mineral}_final_stage_production_tons_{stage_cut}_in_country"
#         if production_col in row:
#             try:
#                 production_amount = float(row[production_col])
#                 demand = (production_amount * 1000) * energy_intensity
#                 total_demand += demand
#                 demand_by_combination[f"demand_{mineral}_{stage_cut}"] = demand
#             except (ValueError, TypeError):
#                 print(f"Warning: Non-numeric value found in column {production_col}")
#                 demand_by_combination[f"demand_{mineral}_{stage_cut}"] = 0
#                 continue
    
#     demand_by_combination["total_demand"] = total_demand
#     return demand_by_combination


In [11]:
# import pandas as pd
# import geopandas as gpd
# import fiona

# # Read the unique values dictionary from the Excel file
# unique_values_mine_gdf = pd.read_excel("min_processing_values_dict.xlsx")
# result_dict = {(row['mineral'], row['stage_cut']): row['processing_energy_intensity'] for index, row in unique_values_mine_gdf.iterrows()}

# # Initialize an empty summary DataFrame
# summary_df = pd.DataFrame()

# # Define the path and name of the GeoPackage file
# mine_path = r"C:\Users\alexl\Dropbox\Self-employment\Imperial work\Mineral Work\GIS_data\MiningData\DataFromRaghavJuly2024"
# mine_name = "node_locations_for_energy_conversion_country_unconstrained.gpkg"

# # List layers in the GeoPackage file
# layers = fiona.listlayers(mine_path + "\\" + mine_name)

# mine_gdf = gpd.read_file(mine_path + "\\" + mine_name, layer=layers[0])

In [15]:
# demand_results = mine_gdf.apply(calculate_demand_country, axis=1)
# demand_results_df = pd.DataFrame(demand_results.tolist())

In [17]:
# # Add the new columns to the mine_gdf
# mine_gdf = pd.concat([mine_gdf, demand_results_df], axis=1)